# **Modelado y entrenamiento**
## **Sistema de clasificación de sonidos urbanos para alertas**

**Proyecto:** Detección de sonidos de emergencia en entornos urbanos  
**Equipo:** Alessandra · Natalia · Andrés

### **¿Qué hace este notebook?**

Este notebook tiene varios objetivos:

1. **Configurar y revisar** el dataset procesado antes de entrenar (EDA rápido del split).
2. **Instanciar** el modelo `DualBranchCNN` (mel + MFCC) con todos sus componentes.
3. **Entrenar** con checkpointing automático y reanudación desde el último checkpoint.
4. **Visualizar** las curvas de entrenamiento en tiempo real después de cada época.
5. **Evaluar** el mejor modelo sobre el conjunto de test con métricas completas.

### **Arquitectura del modelo**

```
Mel  [B,1,128,T] --> FeatureBranch CNN ──┐
                                        cat [B,512] --> Classifier --> [B, num_classes]
MFCC [B,1, 40,T] --> FeatureBranch CNN ──┘
```

Cada `FeatureBranch` contiene 4 bloques Conv-BN-ReLU + ResBlock con Global Average Pooling.

## **1. Dependencias**

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os, time, random, glob, pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13})

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
print(f'PyTorch {torch.__version__}')

## **2. Imports del proyecto**

In [ ]:
from src.utils.config import (
    PROCESSED_METADATA_SPLIT_FIX,
    LABEL_MAPPING,
    CHECKPOINT_DIR,
    FINAL_MODEL_DIR,
)
from src.models.audio_dataset import ProcessedAudioDataset, crnn_collate_fn
from src.models.hybrid_cnn import ImprovedMFCCCNN # DualBranchCNN con alias compatible
from src.models.trainV1 import (
    set_seed,
    build_loaders,
    train_one_epoch,
    evaluate,
    save_checkpoint,
    load_latest_checkpoint,
    prepare_model_inputs,
    CFG,
)

print('Importaciones OK')
print(f'PROCESSED_METADATA: {PROCESSED_METADATA_SPLIT_FIX}')
print(f'CHECKPOINT_DIR: {CHECKPOINT_DIR}')
print(f'FINAL_MODEL_DIR: {FINAL_MODEL_DIR}')

## 3. Configuración del experimento

Modificar `CFG` aquí para cambiar hiperparámetros sin tocar `trainV1.py`.

| Parámetro | Valor | Descripción |
|---|---|---|
| `mode` | `mel_mfcc` | Usa ambas ramas (mel + MFCC). Cambiar a `mel_only` para ablación. |
| `epochs` | 12 | Épocas totales. El training es incremental desde el último checkpoint. |
| `lr` | 3e-4 | Learning rate inicial para AdamW. |
| `label_smoothing` | 0.05 | Suavizado de etiquetas para reducir overconfidence. |
| `target_type` | `human_label` | Cambiar a `alertable` para el modelo binario de alertas. |


In [ ]:
# Ajustar aquí los hiperparámetros
class CFG:
    batch_size = 32
    lr = 3e-4
    weight_decay = 1e-2
    epochs = 12
    num_workers = 0
    use_mfcc = True
    use_scalars = False
    seed = 42
    print_every = 50
    target_type = 'human_label' # 'human_label' | 'alertable'
    mode = 'mel_mfcc' # 'mel_only' | 'mel_mfcc'
    dropout = 0.25
    label_smoothing = 0.05
    val_split = 0.10 # fracción de train usada para validación
    checkpoint_dir = CHECKPOINT_DIR
    save_every = 2 # guardar checkpoint cada número de épocas

cfg = CFG()
set_seed(cfg.seed)

print('Configuración activa:')
for k, v in vars(cfg).items():
    print(f'{k:20s}: {v}')

## **4. EDA del dataset procesado**
- Antes de entrenar, verificamos que el dataset está bien construido y entendemos con qué trabajamos.

### **4.1 Carga y resumen general**

In [ ]:
df = pd.read_csv(PROCESSED_METADATA_SPLIT_FIX)

label_path = LABEL_MAPPING[cfg.target_type]
with open(label_path, 'rb') as f:
    payload = pickle.load(f)
label2idx = payload.get('label2idx', payload)
idx2label = {v: k for k, v in label2idx.items()}
num_classes = len(label2idx)

print(f'Dataset: {len(df):,} muestras | {num_classes} clases | target: {cfg.target_type}')
print(f'Train: {(df.split=="train").sum():,} | Test: {(df.split=="test").sum():,}')
print(f'\nPrimeras filas:')
df.head()

### **4.2 Distribución de clases y desbalanceo**

In [ ]:
class_counts = df[df.split=='train'][cfg.target_type].value_counts()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Top 20 clases
top20 = class_counts.head(20)
axes[0].barh(range(len(top20)), top20.values[::-1],
             color=sns.color_palette('viridis', 20))
axes[0].set_yticks(range(len(top20)))
axes[0].set_yticklabels([str(l)[:25] for l in top20.index[::-1]], fontsize=7)
axes[0].set_title('Top 20 clases (train)')
axes[0].set_xlabel('Muestras')

# Histograma del desbalanceo
axes[1].hist(class_counts.values, bins=40, color='steelblue', edgecolor='white')
axes[1].axvline(class_counts.mean(), color='red', ls='--', label=f'Media: {class_counts.mean():.0f}')
axes[1].axvline(class_counts.median(), color='orange', ls='--', label=f'Mediana: {class_counts.median():.0f}')
axes[1].set_title('Distribución de muestras por clase')
axes[1].set_xlabel('Muestras por clase')
axes[1].set_ylabel('Número de clases')
axes[1].legend(fontsize=8)

# Curva de Lorenz
sorted_c = np.sort(class_counts.values)
cum_s = np.cumsum(sorted_c) / sorted_c.sum()
cum_c = np.arange(1, len(sorted_c)+1) / len(sorted_c)
axes[2].plot(cum_c*100, cum_s*100, 'steelblue', lw=2, label='Dataset')
axes[2].plot([0,100],[0,100],'k--', alpha=0.4, label='Perfecto equilibrio')
axes[2].fill_between(cum_c*100, cum_s*100, np.linspace(0, 100, len(sorted_c)), alpha=0.15, color='steelblue')
axes[2].set_title('Curva de Lorenz (desbalanceo)')
axes[2].set_xlabel('% clases acumuladas')
axes[2].set_ylabel('% muestras acumuladas')
axes[2].legend(fontsize=8)

plt.suptitle(f'Análisis de clases - {cfg.target_type}', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Clases con < 30 muestras en train: {(class_counts < 30).sum()} / {num_classes}')
print(f'Ratio max/min: {class_counts.max() / class_counts.min():.2f}x')

### **4.3 Verificación de los splits train / val / test**

In [ ]:
train_ds, test_ds, train_loader_full, test_loader = build_loaders(cfg)

val_size = int(len(train_ds) * cfg.val_split)
train_size = len(train_ds) - val_size

train_subset, val_subset = random_split(
    train_ds, [train_size, val_size],
    generator=torch.Generator().manual_seed(cfg.seed)
)

train_loader = DataLoader(train_subset, batch_size=cfg.batch_size, shuffle=True,
                          num_workers=cfg.num_workers, collate_fn=crnn_collate_fn,
                          drop_last=True, pin_memory=(DEVICE.type=='cuda'))
val_loader = DataLoader(val_subset, batch_size=cfg.batch_size, shuffle=False,
                        num_workers=cfg.num_workers, collate_fn=crnn_collate_fn,
                        pin_memory=(DEVICE.type=='cuda'))

# Visualización del split
splits = {'Train': train_size, 'Val': val_size, 'Test': len(test_ds)}
total = sum(splits.values())

fig, ax = plt.subplots(figsize=(7, 3))
colors = ['steelblue', 'orange', 'green']
bars = ax.barh([0], [train_size], color=colors[0], height=0.5, label='Train')
ax.barh([0], [val_size], left=[train_size], color=colors[1], height=0.5, label='Val')
ax.barh([0], [len(test_ds)], left=[train_size+val_size], color=colors[2], height=0.5, label='Test')
ax.set_yticks([])
ax.set_xlabel('Muestras')
ax.set_title('Distribución Train / Val / Test')
ax.legend(loc='upper right')
for i, (name, n) in enumerate(splits.items()):
    left = [0, train_size, train_size+val_size][i]
    ax.text(left + n/2, 0, f'{name}\n{n:,}\n({n/total*100:.2f}%)',
            ha='center', va='center', fontsize=9, color='white', fontweight='bold')
plt.tight_layout()
plt.show()

# Verificar batch shape
sample_batch, sample_labels, _ = next(iter(train_loader))
print('\nShapes de un batch de train:')
for k, v in sample_batch.items():
    print(f'{k:10s}: {tuple(v.shape)}')
print(f'labels: {tuple(sample_labels.shape)} dtype={sample_labels.dtype}')
print(f'num_classes: {num_classes}')

## **5. Modelo**

In [ ]:
model = ImprovedMFCCCNN(
    num_classes=num_classes,
    dropout=cfg.dropout,
    mode=cfg.mode,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Modelo: DualBranchCNN (modo={cfg.mode})')
print(f'Total params: {total_params:,}')
print(f'Trainable params: {trainable_params:,}')
print(f'Clases: {num_classes}')

if DEVICE.type == 'cuda':
    mem = torch.cuda.memory_allocated() / 1e6
    print(f'GPU mem (modelo): {mem:.2f} MB')

## **6. Loss, Optimizer y Scheduler**

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.lr,
    weight_decay=cfg.weight_decay,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=2,
)

print('Loss: CrossEntropyLoss (label_smoothing=', cfg.label_smoothing, ')')
print('Optimizer: AdamW (lr=', cfg.lr, 'wd=', cfg.weight_decay,')')
print('Scheduler: ReduceLROnPlateau (patience=2, factor=0.5)')

## **7. Checkpoint: reanudación automática**
- Si ya hay un checkpoint guardado en `CHECKPOINT_DIR`, el entrenamiento **se reanuda automáticamente** desde la última época guardada. Si es la primera vez, empieza desde cero.

In [ ]:
history = {
    'epoch': [], 'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': [],
    'precision': [], 'recall': [], 'f1': [],
    'lr': [], 'epoch_time': [], 'images_per_sec': []
}

start_epoch = 0
best_acc = 0.0
best_epoch = 0

ckpt_epoch, ckpt_history, ckpt_best_acc, ckpt_best_epoch = load_latest_checkpoint(
    cfg, model, optimizer
)

if ckpt_epoch is not None:
    start_epoch = ckpt_epoch
    history = ckpt_history
    best_acc = ckpt_best_acc
    best_epoch = ckpt_best_epoch
    # compatibilidad con checkpoints antiguos
    for key in list(history.keys()):
        if key == 'test_loss' and 'val_loss' not in history:
            history['val_loss'] = history.pop('test_loss')
        if key == 'test_acc' and 'val_acc' not in history:
            history['val_acc'] = history.pop('test_acc')
    for key in ['epoch','train_loss','train_acc','val_loss','val_acc',
                'precision','recall','f1','lr','epoch_time','images_per_sec']:
        if key not in history:
            history[key] = []
    print(f'Reanudando desde época {start_epoch} | mejor val_acc hasta ahora: {best_acc:.4f}')
else:
    print('Entrenamiento desde cero')

epochs_remaining = cfg.epochs - start_epoch
print(f'Épocas restantes: {epochs_remaining} / {cfg.epochs}')

## **8. Entrenamiento**
- **Tiempo estimado:** ~3-5 min por época en GPU, ~15-30 min por época en CPU
- Se guarda un checkpoint cada `save_every=2` épocas y el mejor modelo en `FINAL_MODEL_DIR/best_mfcc_cnn.pt`

In [ ]:
print('=' * 70)
print(f'INICIANDO ENTRENAMIENTO')
print(f'Épocas: {start_epoch+1} --> {cfg.epochs}')
print(f'Modo: {cfg.mode} | Target: {cfg.target_type} | Clases: {num_classes}')
print('=' * 70)

for epoch in range(start_epoch + 1, cfg.epochs + 1):

    t0 = time.time()
    current_lr = optimizer.param_groups[0]['lr']

    print(f'\nEpoch {epoch}/{cfg.epochs} | LR: {current_lr:.2e}')
    print('=' * 70)

    # TRAIN
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, DEVICE, cfg, epoch
    )

    # VALIDACIÓN
    val_loss, val_acc, precision, recall, f1 = evaluate(
        model, val_loader, criterion, DEVICE, cfg, epoch
    )

    scheduler.step(val_acc)

    dt = time.time() - t0
    imgs_sec = len(train_subset) / dt

    # HISTORY
    history['epoch'].append(epoch)
    history['train_loss'].append(float(train_loss))
    history['train_acc'].append(float(train_acc))
    history['val_loss'].append(float(val_loss))
    history['val_acc'].append(float(val_acc))
    history['precision'].append(float(precision))
    history['recall'].append(float(recall))
    history['f1'].append(float(f1))
    history['lr'].append(float(current_lr))
    history['epoch_time'].append(float(dt))
    history['images_per_sec'].append(float(imgs_sec))

    # LOG
    print(f'\nTrain --> Loss: {train_loss:.4f} | Acc: {train_acc:.4f}')
    print(f'Val --> Loss: {val_loss:.4f} | Acc: {val_acc:.4f}')
    print(f'Metrics --> P: {precision:.4f} | R: {recall:.4f} | F1: {f1:.4f}')
    print(f'Epoch time: {dt:.2f}s | Imágenes por segundo: {imgs_sec:.0f} img/s')

    # EL MEJOR MODELO
    if val_acc > best_acc:
        best_acc = val_acc
        best_epoch = epoch
        os.makedirs(FINAL_MODEL_DIR, exist_ok=True)
        torch.save(model.state_dict(), FINAL_MODEL_DIR / 'best_mfcc_cnn.pt')
        print(f'Nuevo mejor modelo guardado (val_acc={best_acc:.4f})')

    # CHECKPOINT
    if epoch % cfg.save_every == 0:
        save_checkpoint(cfg, model, optimizer, epoch, history, best_acc, best_epoch)

print(f'\nEntrenamiento completado.')
print(f'Mejor época: {best_epoch} | Mejor val_acc: {best_acc:.4f}')

## **9. Curvas de entrenamiento**

In [ ]:
epochs_range = history['epoch']

fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.3)

# Loss
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(epochs_range, history['train_loss'], 'o-', label='Train', color='steelblue')
ax1.plot(epochs_range, history['val_loss'], 'o-', label='Val', color='salmon')
ax1.set_title('Loss')
ax1.set_xlabel('Época')
ax1.set_ylabel('Loss')
ax1.legend()

# Accuracy
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(epochs_range, history['train_acc'], 'o-', label='Train', color='steelblue')
ax2.plot(epochs_range, history['val_acc'], 'o-', label='Val', color='salmon')
if best_epoch in epochs_range:
    bi = epochs_range.index(best_epoch)
    ax2.axvline(best_epoch, color='green', ls='--', alpha=0.6, label=f'Best (e{best_epoch})')
    ax2.scatter([best_epoch], [history['val_acc'][bi]], color='green', zorder=5, s=80)
ax2.set_title('Accuracy')
ax2.set_xlabel('Época')
ax2.set_ylabel('Accuracy')
ax2.legend()

# F1 / Precision / Recall
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(epochs_range, history['f1'], 'o-', label='F1', color='green')
ax3.plot(epochs_range, history['precision'], 's--', label='Precision', color='red')
ax3.plot(epochs_range, history['recall'], '^--', label='Recall', color='blue')
ax3.set_title('F1 / Precision / Recall (val, weighted)')
ax3.set_xlabel('Época')
ax3.legend(fontsize=8)

# Learning rate
ax4 = fig.add_subplot(gs[1, 0])
ax4.plot(epochs_range, history['lr'], 'o-', color='brown')
ax4.set_title('Learning rate')
ax4.set_xlabel('Época')
ax4.set_ylabel('Learning rate')
ax4.set_yscale('log')

# Throughput
ax5 = fig.add_subplot(gs[1, 1])
ax5.bar(epochs_range, history['images_per_sec'], color='steelblue', edgecolor='white')
ax5.set_title('Throughput (img/s)')
ax5.set_xlabel('Época')
ax5.set_ylabel('img/s')

# Tiempo por época
ax6 = fig.add_subplot(gs[1, 2])
ax6.bar(epochs_range, history['epoch_time'], color='orange', edgecolor='white')
ax6.set_title('Tiempo por época (s)')
ax6.set_xlabel('Época')
ax6.set_ylabel('Segundos')

plt.suptitle(f'Curvas de entrenamiento | {cfg.mode} | {cfg.target_type}', fontsize=14)
plt.savefig(FINAL_MODEL_DIR / 'training_curves.png', bbox_inches='tight', dpi=150)
plt.show()
print('Gráfico guardado en FINAL_MODEL_DIR/training_curves.png')

### **9.1 Diagnóstico de overfitting**

In [ ]:
gap_loss = [tl - vl for tl, vl in zip(history['train_loss'], history['val_loss'])]
gap_acc = [va - ta for ta, va in zip(history['train_acc'],  history['val_acc'])]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(epochs_range, gap_loss,
            color=['red' if g > 0 else 'green' for g in gap_loss], edgecolor='white')
axes[0].axhline(0, color='black', lw=0.8)
axes[0].set_title('Loss gap (train − val)\n> 0 = generalizando bien')
axes[0].set_xlabel('Época')

axes[1].bar(epochs_range, gap_acc,
            color=['red' if g < 0 else 'green' for g in gap_acc], edgecolor='white')
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_title('Accuracy gap (val − train)\n< 0 = overfitting')
axes[1].set_xlabel('Época')

plt.tight_layout()
plt.show()

last_gap = gap_acc[-1] if gap_acc else 0
if last_gap < -0.10:
    print(f'Posible overfitting: val_acc - train_acc = {last_gap:.4f}')
    print('Considerar: más dropout, más epochs con lr menor o data augmentation.')
elif last_gap > 0.05:
    print(f'Val > Train ({last_gap:.4f}): el modelo puede seguir mejorando; sigamos entrenando.')
else:
    print(f'Gap saludable ({last_gap:.4f}): buen equilibrio bias-varianza.')

## **10. Evaluación final en test**
- Cargamos el **mejor modelo** guardado durante el entrenamiento y lo evaluamos sobre el conjunto de test.

In [ ]:
best_model = ImprovedMFCCCNN(
    num_classes=num_classes,
    dropout=cfg.dropout,
    mode=cfg.mode,
).to(DEVICE)

best_model.load_state_dict(
    torch.load(FINAL_MODEL_DIR / 'best_mfcc_cnn.pt', map_location=DEVICE)
)

test_loss, test_acc, precision, recall, f1 = evaluate(
    best_model, test_loader, criterion, DEVICE, cfg, epoch='TEST'
)

print('\n' + '='*60)
print('RESULTADOS FINALES EN TEST')
print('='*60)
print(f'Test Loss: {test_loss:.4f}')
print(f'Test Acc: {test_acc:.4f} | ({test_acc*100:.2f}%)')
print(f'Precision: {precision:.4f} (weighted)')
print(f'Recall: {recall:.4f} (weighted)')
print(f'F1 Score: {f1:.4f} (weighted)')
print('='*60)
print(f'Mejor época: {best_epoch} | Mejor val_acc: {best_acc:.4f}')

### **10.1 Matriz de confusión y clases más difíciles**

In [ ]:
@torch.no_grad()
def get_predictions(model, loader, device, cfg):
    model.eval()
    y_true, y_pred = [], []
    for batch, labels, _ in loader:
        inputs = prepare_model_inputs(batch, device, cfg.mode)
        outputs = model(*inputs)
        preds = outputs.argmax(dim=1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())
    return np.array(y_true), np.array(y_pred)

y_true, y_pred = get_predictions(best_model, test_loader, DEVICE, cfg)
print(f'Predicciones obtenidas: {len(y_true):,} muestras')

In [ ]:
# Clases presentes en test (puede no haber todas)
present_classes = sorted(np.unique(np.concatenate([y_true, y_pred])))
present_labels  = [str(idx2label.get(i, i))[:20] for i in present_classes]

cm = confusion_matrix(y_true, y_pred, labels=present_classes)

# Mostrar solo si hay pocas clases; de lo contrario, mostrar top errores
if len(present_classes) <= 30:
    fig, ax = plt.subplots(figsize=(max(10, len(present_classes)//2),
                                    max(8, len(present_classes)//2)))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=present_labels, yticklabels=present_labels,
                linewidths=0.5, ax=ax)
    ax.set_xlabel('Predicho')
    ax.set_ylabel('Real')
    ax.set_title('Matriz de confusión (test)')
    plt.tight_layout()
    plt.show()
else:
    print(f'{len(present_classes)} clases: mostrando un resumen en lugar de la matriz completa.')

In [ ]:
# Top 15 clases con peor accuracy (más errores)
per_class_acc = cm.diagonal() / cm.sum(axis=1).clip(min=1)
worst_idx = np.argsort(per_class_acc)[:15]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Peores clases
worst_labels = [present_labels[i] for i in worst_idx]
worst_accs = per_class_acc[worst_idx]
axes[0].barh(range(len(worst_idx)), worst_accs[::-1],
             color=sns.color_palette('Reds_r', len(worst_idx)))
axes[0].set_yticks(range(len(worst_idx)))
axes[0].set_yticklabels(worst_labels[::-1], fontsize=8)
axes[0].set_xlabel('Accuracy por clase')
axes[0].set_title('Top 15 clases con peor accuracy')
axes[0].axvline(0.5, color='red', ls='--', alpha=0.5)

# Mejores clases
best_idx = np.argsort(per_class_acc)[-15:][::-1]
best_labels = [present_labels[i] for i in best_idx]
best_accs = per_class_acc[best_idx]
axes[1].barh(range(len(best_idx)), best_accs[::-1],
             color=sns.color_palette('Greens_r', len(best_idx)))
axes[1].set_yticks(range(len(best_idx)))
axes[1].set_yticklabels(best_labels[::-1], fontsize=8)
axes[1].set_xlabel('Accuracy por clase')
axes[1].set_title('Top 15 clases con mejor accuracy')

plt.suptitle('Análisis por clase (test)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Classification report completo (top 20 clases más frecuentes en test)
top_test_classes = pd.Series(y_true).value_counts().head(20).index.tolist()
top_labels_str = [str(idx2label.get(i, i)) for i in top_test_classes]

report = classification_report(
    y_true, y_pred,
    labels=top_test_classes,
    target_names=top_labels_str,
    zero_division=0
)
print('Classification report (Top 20 clases por frecuencia en test):')
print(report)

## **11. Resumen**

In [ ]:
print('=' * 60)
print('RESUMEN')
print('=' * 60)
print(f'Modelo: DualBranchCNN ({cfg.mode})')
print(f'Target: {cfg.target_type}')
print(f'Clases: {num_classes}')
print(f'Épocas: {cfg.epochs} | (mejor: {best_epoch})')
print(f'Batch size: {cfg.batch_size}')
print(f'LR inicial: {cfg.lr}')
print(f'Dropout: {cfg.dropout}')
print(f'Device: {DEVICE}')
print()
print(f'RESULTADOS TEST')
print(f'Accuracy: {test_acc:.4f} | ({test_acc*100:.2f}%)')
print(f'F1 (wgt): {f1:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print()
print(f'Modelo guardado en:')
print(f'{FINAL_MODEL_DIR / "best_mfcc_cnn.pt"}')
print('=' * 60)
print()
print('Continúa leyendo 04.1_evaluation.ipynb para un análisis más profundo.')